# GRIDPILOT AI — Member 2 — Task 5
## LightGBM Demand Forecasting (15/30/60 min)

**Purpose:** Train, evaluate, and version one LightGBM model per forecast horizon, compare against the Chunk 4 baselines on the same test set, and export reproducible model artifacts.


## 0. Setup

In [ ]:
!pip -q install pandas numpy scikit-learn lightgbm pyarrow

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Libraries loaded. LightGBM version:", lgb.__version__)

## 1. Load feature-engineered dataset

In [ ]:
FEATURES_PATH = "./feature_engineering_outputs/demand_features.parquet"

df = pd.read_parquet(FEATURES_PATH)
df = df.sort_values("timestamp").reset_index(drop=True)
DEMAND_COL = "demand" if "demand" in df.columns else df.columns[1]
print(df.shape)

## 2. Chronological split (same convention as the baselines)

In [ ]:
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(len(train_df), len(val_df), len(test_df))

## 3. Define feature set and targets

Targets are built per horizon by shifting the demand column forward -- this is the only place a *future* value is intentionally referenced, and only as the label, never as a feature.

In [ ]:
FEATURE_VERSION = "v1"
FREQ_MINUTES = None  # set from Task 1 / Chunk 3
assert FREQ_MINUTES is not None, "Set FREQ_MINUTES before running."

HORIZONS_MINUTES = [15, 30, 60]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ("timestamp", DEMAND_COL)
    and df[c].dtype != "object"
]
print("Feature columns:", FEATURE_COLS)

def build_targets(data, target_col, horizon_minutes, freq_minutes):
    periods = int(horizon_minutes / freq_minutes)
    return data[target_col].shift(-periods)

for h in HORIZONS_MINUTES:
    for split in (train_df, val_df, test_df):
        split[f"target_{h}min"] = build_targets(split, DEMAND_COL, h, FREQ_MINUTES)

## 4. Training configuration (reproducible)

In [ ]:
LGBM_PARAMS = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "seed": 42,
    "verbosity": -1,
}
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30

print(json.dumps(LGBM_PARAMS, indent=2))

## 5. Train one model per horizon

In [ ]:
models = {}
training_metadata = {}

for h in HORIZONS_MINUTES:
    target_col = f"target_{h}min"

    train_valid = train_df.dropna(subset=FEATURE_COLS + [target_col])
    val_valid = val_df.dropna(subset=FEATURE_COLS + [target_col])

    train_set = lgb.Dataset(train_valid[FEATURE_COLS], label=train_valid[target_col])
    val_set = lgb.Dataset(val_valid[FEATURE_COLS], label=val_valid[target_col], reference=train_set)

    start = time.time()
    model = lgb.train(
        LGBM_PARAMS,
        train_set,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
    )
    elapsed = time.time() - start

    models[h] = model
    training_metadata[h] = {
        "horizon_minutes": h,
        "best_iteration": model.best_iteration,
        "training_seconds": round(elapsed, 2),
        "n_train_rows": len(train_valid),
        "n_val_rows": len(val_valid),
        "feature_version": FEATURE_VERSION,
        "params": LGBM_PARAMS,
    }
    print(f"Horizon {h}min: best_iteration={model.best_iteration}, trained in {elapsed:.1f}s")

## 6. Evaluate on the held-out test set

In [ ]:
def evaluate(y_true, y_pred):
    mask = y_true.notna() & pd.Series(y_pred, index=y_true.index).notna()
    y_true_m = y_true[mask]
    y_pred_m = pd.Series(y_pred, index=y_true.index)[mask]
    mae = mean_absolute_error(y_true_m, y_pred_m)
    rmse = mean_squared_error(y_true_m, y_pred_m, squared=False)
    nonzero = y_true_m != 0
    mape = (np.abs((y_true_m[nonzero] - y_pred_m[nonzero]) / y_true_m[nonzero])).mean() * 100 if nonzero.any() else None
    return {"mae": mae, "rmse": rmse, "mape": mape, "n": int(mask.sum())}

lgbm_results = []
for h in HORIZONS_MINUTES:
    target_col = f"target_{h}min"
    test_valid = test_df.dropna(subset=FEATURE_COLS)
    preds = models[h].predict(test_valid[FEATURE_COLS])
    metrics = evaluate(test_df.loc[test_valid.index, target_col], preds)
    lgbm_results.append({"model": "lightgbm", "horizon_minutes": h, **metrics})

lgbm_results_df = pd.DataFrame(lgbm_results)
display(lgbm_results_df)

## 7. Compare against baselines

Load the baseline evaluation artifact from Chunk 4 and compare on the same metrics.

In [ ]:
BASELINE_RESULTS_PATH = "./baseline_outputs/baseline_evaluation.csv"

try:
    baseline_df = pd.read_csv(BASELINE_RESULTS_PATH)
    comparison = pd.concat([baseline_df, lgbm_results_df], ignore_index=True)
    display(comparison.sort_values(["horizon_minutes", "model"]))
except FileNotFoundError:
    print("Run Chunk 4 (Baselines) first to produce a comparison.")

## 8. Save model artifacts, metadata, and validation metrics

In [ ]:
MODEL_VERSION = "demand-lgbm-v1"
OUTPUT_DIR = Path("./ml_models_demand")
OUTPUT_DIR.mkdir(exist_ok=True)

for h, model in models.items():
    model_path = OUTPUT_DIR / f"{MODEL_VERSION}_h{h}min.txt"
    model.save_model(str(model_path))
    training_metadata[h]["model_version"] = MODEL_VERSION
    training_metadata[h]["model_path"] = str(model_path)

with open(OUTPUT_DIR / f"{MODEL_VERSION}_training_metadata.json", "w") as f:
    json.dump(training_metadata, f, indent=2, default=str)

lgbm_results_df.to_csv(OUTPUT_DIR / f"{MODEL_VERSION}_test_metrics.csv", index=False)

print("Saved model artifacts to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

## Important — claim discipline

Only state the LightGBM model outperforms the baselines if the comparison table in section 7 actually shows lower MAE/RMSE/MAPE on the test set, for each horizon. If it does not beat a baseline at some horizon, report that honestly rather than adjusting the framing.

## Next step

Port the trained model + `forecastDemand` inference logic into `services/forecasting/**` (Chunk 6), returning the shared `DemandForecast` contract.